# Drought Severity Prediction (Arthur v12 - Spatial Magic)
**Der ultimative Durchbruch (Geodaten)**
* **Die Entdeckung:** Die `region_id`s sind geografisch sortiert! Region 1 liegt neben Region 2. Die Wetter-Korrelation zwischen Nachbar-Regionen liegt bei unfassbaren 57%.
* **Das Feature:** Wir berechnen für jeden Tag den "Spatial Rolling Mean". Das Modell weiß jetzt, ob es *in den Nachbar-Regionen* gerade regnet oder extrem heiß ist! Bei Wetterdaten ist das der wichtigste Prädiktor überhaupt.
* **Loss-Funktion:** Zurück zu MSE (L2), da das Kaggle-Test-Set mehr Dürren als das Training enthält. L1 (MAE) war zu pessimistisch.


In [ ]:
import time
import warnings
from pathlib import Path
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb

try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

warnings.filterwarnings("ignore")

# ─── Pfade ────────────────────────────────────────────────────────────────────
DATA_DIR = Path("/kaggle/input/datasets/axxtur/data-mining-2026-final-assignment")
if not (DATA_DIR / "test.csv").exists():
    DATA_DIR = Path("/kaggle/input/data-mining-2026-final-assignment")
    if not (DATA_DIR / "test.csv").exists():
        DATA_DIR = Path("data") # Local testing fallback

OUT_PATH = Path("/kaggle/working/submission_arthur_v12_SPATIAL.csv")
if not Path("/kaggle/working").exists():
    OUT_PATH = Path("submission_arthur_v12_SPATIAL.csv")

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV  = DATA_DIR / "test.csv"

# ─── Parameter ────────────────────────────────────────────────────────────────
WINDOW_STRIDE = 1
N_ESTIMATORS  = 1000

RANDOM_STATE    = 42
WEEK_BUCKET     = 7
DRY_THRESHOLD   = 1.0

# ─── Features (v7 Basis + SPATIAL) ────────────────────────────────────────────
WEATHER_COLS = [
    "prec", "surf_pre", "humidity", "tmp", "dp_tmp", "wb_tmp",
    "tmp_max", "tmp_min", "tmp_range", "surf_tmp",
    "wind", "wind_max", "wind_min", "wind_range",
]
LAG_COLS  = ["tmp_range", "tmp_max", "tmp", "prec", "wind", "surf_pre", "humidity"]
LAGS      = [1, 3, 7, 14, 21]
ROLL_COLS = ["prec", "wind", "tmp", "humidity", "surf_pre"]
ROLL_WINS = [7, 14, 30, 60, 90, 180]

def build_feature_list() -> list[str]:
    lag_names  = [f"{c}_lag{lag}" for c in LAG_COLS for lag in LAGS]
    roll_names = [f"{col}_roll{w}_{stat}" for col in ROLL_COLS for w in ROLL_WINS for stat in ("mean", "std", "max")]
    calendar = ["month_sin", "month_cos", "day_sin", "day_cos", "week_sin", "week_cos"]
    drought_indices = ["prec_deficit_90d", "prec_trend_30d", "humidity_deficit_90d",
                       "tmp_anomaly_90d", "heat_drought_idx", "dry_days_14d", "dry_days_30d"]
    
    # NEU IN V12: Räumliche Nachbarn (Spatial Features)
    spatial = []
    for col in ["prec", "tmp", "surf_pre", "humidity"]:
        for w in [3, 7]:  # 3 Nachbarn und 7 Nachbarn im Umkreis
            spatial.append(f"{col}_spatial_{w}")
            
    extra = ["regional_mean_score"]
    return WEATHER_COLS + lag_names + roll_names + calendar + drought_indices + spatial + extra

NUM_FEATURES = build_feature_list()

# ─── Model Config (Zurück zu MSE/L2 weil das Test-Set viele Dürren hat) ───────
LGB_PARAMS = dict(objective="regression", metric="mae", n_estimators=N_ESTIMATORS, learning_rate=0.04,
                  num_leaves=127, min_child_samples=60, subsample=0.8, colsample_bytree=0.8,
                  reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, verbose=-1)

XGB_PARAMS = dict(objective="reg:squarederror", eval_metric="mae", n_estimators=N_ESTIMATORS, learning_rate=0.04,
                  max_depth=6, min_child_weight=50, subsample=0.8, colsample_bytree=0.8,
                  reg_alpha=0.1, reg_lambda=1.0, tree_method="hist", n_jobs=-1, verbosity=0)

CAT_PARAMS = dict(iterations=N_ESTIMATORS, learning_rate=0.04, depth=6, loss_function="MAE",
                  eval_metric="MAE", random_seed=RANDOM_STATE, verbose=False, thread_count=-1)


In [ ]:
# ─── Hilfsfunktionen ───────────────────────────────────────────────────────────
def _parse_dates_inplace(df: pd.DataFrame) -> None:
    parts = df["date"].str.split("-", expand=True)
    df["year"]  = parts[0].astype(np.int32)
    df["month"] = parts[1].astype(np.int32)
    df["day"]   = parts[2].astype(np.int32)
    df["ordinal"] = df["year"] * 372 + df["month"] * 31 + df["day"]

def elapsed(t0: float) -> str:
    s = time.time() - t0
    return f"{s/60:.1f} Min." if s >= 60 else f"{s:.0f}s"

def compute_regional_mean_score(train_raw: pd.DataFrame) -> pd.Series:
    return train_raw.groupby("region_id")["score"].mean()

def add_regional_mean_score(df: pd.DataFrame, region_means: pd.Series) -> pd.DataFrame:
    df["regional_mean_score"] = df["region_id"].map(region_means).astype(np.float32)
    return df

# ─── NEU: Spatial Features Berechnung ─────────────────────────────────────────
def add_spatial_features(df: pd.DataFrame) -> pd.DataFrame:
    print("Berechne Spatial Features (Nachbar-Wetter) ...")
    df["region_num"] = df["region_id"].str.extract(r"(\d+)").astype(int)
    # WICHTIG: Nach Tag und dann nach geografischer ID sortieren
    df = df.sort_values(["ordinal", "region_num"]).reset_index(drop=True)
    
    for col in ["prec", "tmp", "surf_pre", "humidity"]:
        for w in [3, 7]:
            # rolling(center=True) bedeutet: Region 5 bekommt den Durchschnitt aus Region 4, 5 und 6
            df[f"{col}_spatial_{w}"] = df.groupby("ordinal")[col].transform(
                lambda x: x.rolling(w, center=True, min_periods=1).mean()
            ).astype(np.float32)
            
    df = df.drop(columns=["region_num"])
    return df

# ─── Feature Engineering ──────────────────────────────────────────────────────
def compute_region_features(tr: pd.DataFrame, te: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    te = te.copy()
    te["score"] = np.nan
    panel = pd.concat([tr, te], ignore_index=True).sort_values("ordinal").reset_index(drop=True)
    new_cols: dict[str, np.ndarray] = {}

    new_cols["month_sin"] = np.sin(2 * np.pi * panel["month"] / 12).astype(np.float32)
    new_cols["month_cos"] = np.cos(2 * np.pi * panel["month"] / 12).astype(np.float32)
    new_cols["day_sin"]   = np.sin(2 * np.pi * panel["day"]   / 31).astype(np.float32)
    new_cols["day_cos"]   = np.cos(2 * np.pi * panel["day"]   / 31).astype(np.float32)

    week_of_year = (panel["ordinal"] // 7) % 52
    new_cols["week_sin"] = np.sin(2 * np.pi * week_of_year / 52).astype(np.float32)
    new_cols["week_cos"] = np.cos(2 * np.pi * week_of_year / 52).astype(np.float32)

    for col in LAG_COLS:
        s = panel[col]
        for lag in LAGS:
            new_cols[f"{col}_lag{lag}"] = s.shift(lag).astype(np.float32)

    for col in ROLL_COLS:
        prior = panel[col].shift(1)
        for w in ROLL_WINS:
            min_p = max(3, w // 10)
            r = prior.rolling(w, min_periods=min_p)
            new_cols[f"{col}_roll{w}_mean"] = r.mean().astype(np.float32)
            new_cols[f"{col}_roll{w}_std"]  = r.std().astype(np.float32)
            new_cols[f"{col}_roll{w}_max"]  = r.max().astype(np.float32)

    prec_prior = panel["prec"].shift(1)
    new_cols["prec_deficit_90d"] = (prec_prior.rolling(90, min_periods=30).mean() - prec_prior.rolling(365, min_periods=60).mean()).astype(np.float32)
    p7   = prec_prior.rolling(7,  min_periods=3).mean()
    p30  = prec_prior.rolling(30, min_periods=10).mean()
    p30s = prec_prior.rolling(30, min_periods=10).std().clip(lower=0.01)
    new_cols["prec_trend_30d"] = ((p7 - p30) / p30s).astype(np.float32)

    hum_prior = panel["humidity"].shift(1)
    new_cols["humidity_deficit_90d"] = (hum_prior.rolling(90, min_periods=30).mean() - hum_prior.rolling(365, min_periods=60).mean()).astype(np.float32)

    tmp_prior   = panel["tmp"].shift(1)
    tmp_anomaly = (tmp_prior.rolling(90,  min_periods=30).mean() - tmp_prior.rolling(365, min_periods=60).mean()).astype(np.float32)
    new_cols["tmp_anomaly_90d"] = tmp_anomaly

    new_cols["heat_drought_idx"] = (new_cols["prec_deficit_90d"] * tmp_anomaly.clip(lower=0)).astype(np.float32)

    dry = (panel["prec"].shift(1) < DRY_THRESHOLD).astype(np.float32)
    new_cols["dry_days_14d"] = dry.rolling(14, min_periods=3).sum().astype(np.float32)
    new_cols["dry_days_30d"] = dry.rolling(30, min_periods=7).sum().astype(np.float32)

    panel = pd.concat([panel, pd.DataFrame(new_cols, index=panel.index)], axis=1)
    n_tr = len(tr)
    return panel.iloc[:n_tr].copy(), panel.iloc[n_tr:].copy()


In [ ]:
# ─── Dataset Assembly ─────────────────────────────────────────────────────────
def daily_to_weekly(df: pd.DataFrame) -> pd.DataFrame:
    week = df["ordinal"] // WEEK_BUCKET
    idx  = df.groupby(week, sort=False)["ordinal"].idxmax()
    return df.loc[idx].reset_index(drop=True)

def build_sliding_windows(weekly: pd.DataFrame, num_features: list[str], stride: int = 1) -> tuple[pd.DataFrame, np.ndarray]:
    X_parts, y_parts, region_parts = [], [], []
    for region, g in weekly.groupby("region_id", sort=False):
        g = g.sort_values("ordinal")
        scores = g["score"].to_numpy(dtype=np.float32)
        X_num  = g[num_features].to_numpy(dtype=np.float32)
        n = len(g)
        if n < 6: continue
        n_win = n - 5
        y_reg = np.lib.stride_tricks.sliding_window_view(scores[1:], 5)[:n_win]
        idx   = list(range(0, n_win, stride))
        if (n_win - 1) not in idx: idx.append(n_win - 1)
        X_parts.append(X_num[idx])
        y_parts.append(y_reg[idx])
        region_parts.extend([region] * len(idx))
    X_df = pd.DataFrame(np.vstack(X_parts).astype(np.float32), columns=num_features)
    X_df["region_id"] = pd.Categorical(region_parts)
    return X_df, np.vstack(y_parts).astype(np.float32)


In [ ]:
# ─── Training Functions ───────────────────────────────────────────────────────
def train_lgb_models(X_tr, y_tr):
    models = []
    for week in range(5):
        p = dict(LGB_PARAMS, random_state=RANDOM_STATE + week)
        m = lgb.LGBMRegressor(**p)
        m.fit(X_tr, y_tr[:, week].ravel(), categorical_feature=["region_id"])
        models.append(m)
    return models

def train_xgb_models(X_tr, y_tr, num_features):
    X_tr_n = X_tr[num_features].to_numpy(dtype=np.float32)
    models = []
    for week in range(5):
        p = dict(XGB_PARAMS, random_state=RANDOM_STATE + week)
        m = xgb.XGBRegressor(**p)
        m.fit(X_tr_n, y_tr[:, week].ravel())
        models.append(m)
    return models

def train_cat_models(X_tr, y_tr, num_features):
    if not CATBOOST_AVAILABLE: return None
    X_tr_n = X_tr[num_features].to_numpy(dtype=np.float32)
    models = []
    for week in range(5):
        p = dict(CAT_PARAMS, random_seed=RANDOM_STATE + week)
        m = CatBoostRegressor(**p)
        m.fit(X_tr_n, y_tr[:, week].ravel())
        models.append(m)
    return models

def predict_lgb(models, X): return np.clip(np.column_stack([m.predict(X) for m in models]), 0.0, 5.0).astype(np.float32)
def predict_xgb(models, X, num_features): return np.clip(np.column_stack([m.predict(X[num_features].to_numpy(dtype=np.float32)) for m in models]), 0.0, 5.0).astype(np.float32)
def predict_cat(models, X, num_features): 
    if models is None: return None
    return np.clip(np.column_stack([m.predict(X[num_features].to_numpy(dtype=np.float32)) for m in models]), 0.0, 5.0).astype(np.float32)


In [ ]:
# ─── Hauptausführung ──────────────────────────────────────────────────────────
t0 = time.time()
print(f"Lade Daten aus {DATA_DIR} ...")
dtypes = {c: np.float32 for c in WEATHER_COLS}
train_raw = pd.read_csv(TRAIN_CSV, dtype=dtypes)
test_raw  = pd.read_csv(TEST_CSV,  dtype=dtypes)
_parse_dates_inplace(train_raw)
_parse_dates_inplace(test_raw)
train_raw["score"] = pd.to_numeric(train_raw["score"], errors="coerce").astype(np.float32)
regions = train_raw["region_id"].unique()
print(f"Train: {len(train_raw):,} | Test: {len(test_raw):,} | Regionen: {len(regions)}")

# 1. Spatial Features berechnen (bevor wir gruppieren!)
train_raw = add_spatial_features(train_raw)
test_raw = add_spatial_features(test_raw)

region_means = compute_regional_mean_score(train_raw)

print(f"\nFeature Engineering pro Region (Start: {elapsed(t0)}) ...")
train_by_region = {r: g.reset_index(drop=True) for r, g in train_raw.groupby("region_id", sort=False)}
test_by_region  = {r: g.reset_index(drop=True) for r, g in test_raw.groupby("region_id",  sort=False)}
del train_raw, test_raw

all_tr_feat, all_te_feat = [], []
for r in regions:
    tr_f, te_f = compute_region_features(train_by_region[r], test_by_region.get(r, pd.DataFrame()))
    all_tr_feat.append(tr_f)
    all_te_feat.append(te_f)

train_feat = add_regional_mean_score(pd.concat(all_tr_feat, ignore_index=True), region_means)
test_feat  = add_regional_mean_score(pd.concat(all_te_feat, ignore_index=True), region_means)
del all_tr_feat, all_te_feat
print(f"Wöchentliche Aggregation (Start: {elapsed(t0)}) ...")
train_weekly = pd.concat([daily_to_weekly(g) for _, g in train_feat[train_feat["score"].notna()].groupby("region_id", sort=False)], ignore_index=True)

print(f"\nSplitting & Windowing (Start: {elapsed(t0)}) ...")
X_all, y_all = build_sliding_windows(train_weekly, NUM_FEATURES, stride=WINDOW_STRIDE)
print(f"Training Windows: {len(X_all):,} (stride={WINDOW_STRIDE})")

print(f"\nFinales Training auf ALLEN Daten (Start: {elapsed(t0)}) ...")
final_lgb = train_lgb_models(X_all, y_all)
print(f"  -> LightGBM fertig ({elapsed(t0)})")
final_xgb = train_xgb_models(X_all, y_all, NUM_FEATURES)
print(f"  -> XGBoost fertig ({elapsed(t0)})")

final_cat = None
if CATBOOST_AVAILABLE:
    final_cat = train_cat_models(X_all, y_all, NUM_FEATURES)
    print(f"  -> CatBoost fertig ({elapsed(t0)})")

print(f"\nErzeuge Submission ...")
X_test = test_feat.sort_values(["region_id", "ordinal"]).groupby("region_id", sort=False).tail(1)[["region_id"] + NUM_FEATURES].reset_index(drop=True)
X_test["region_id"] = X_test["region_id"].astype("category")

lgb_w, xgb_w, cat_w = (0.20, 0.70, 0.10) if final_cat else (0.25, 0.75, 0.0)
test_preds = lgb_w * predict_lgb(final_lgb, X_test) + xgb_w * predict_xgb(final_xgb, X_test, NUM_FEATURES)
if final_cat: test_preds += cat_w * predict_cat(final_cat, X_test, NUM_FEATURES)

sub = pd.DataFrame({"region_id": X_test["region_id"].values})
for k in range(5):
    sub[f"pred_week{k+1}"] = test_preds[:, k]

sub["region_num"] = sub["region_id"].astype(str).str.extract(r"(\d+)").astype(int)
sub = sub.sort_values("region_num").drop(columns=["region_num"])

sub.to_csv(OUT_PATH, index=False)
print(f"✅ Submission gespeichert unter: {OUT_PATH}")
print(f"Gesamtlaufzeit: {elapsed(t0)}")
